# **CUHK-STAT3009: Final Quiz 2025** (100 points)

**Rules and Guidelines:**

- You may use lecture notes, Jupyter notebooks, and online resources from the [whitelist](https://hackmd.io/@cTHE1QvtQuKo2qiAA5u0dA/rJqzNb1Z-x) to complete the quiz.
- You must work independently and must not communicate with others during the quiz.
- AI-powered tools such as ChatGPT, Claude, Gemini, and similar platforms are strictly prohibited.
- Social media and messaging platforms (e.g., WhatsApp, WeChat) are not permitted.

**Academic Integrity:**

Violating these rules constitutes academic dishonesty and will result in a score of zero on the quiz.

> I have read and understood the rules. To confirm, I will:
>
> 1. Read and follow the [Final Quiz Instructions](https://hackmd.io/TbSfC1NLSuuhcoCSYoLVFg?view).
> 2. Complete the quiz independently to the best of my ability.

# **Q1: Recommendation System Evaluation Metrics** (10 points)

## **Q1.1: Root Mean Squared Logarithmic Error** (5 points)

The Root Mean Squared Logarithmic Error (RMSLE) is useful when ratings span different scales. It is defined as:

$$
\text{RMSLE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} \left(\log(1 + r_i) - \log(1 + \hat{r}_i)\right)^2}
$$

where $r_i$ is the actual rating and $\hat{r}_i$ is the predicted rating. The logarithm is the natural logarithm.

**Task:**

Implement `RMSLE(y_true, y_pred)` and use it to compute the RMSLE for the example below.

```python
import numpy as np

y_test = np.array([5.0, 4.0, 3.0, 4.5, 2.5])
y_pred = np.array([4.5, 4.2, 3.5, 4.0, 2.8])
```

In [ ]:
## Your solution here.

## **Q1.2: Weighted RMSLE** (5 points)

In some scenarios, certain ratings are more important than others. For example, higher ratings might carry more weight in the evaluation.

The weighted RMSLE is defined as:

$$
\text{WRMSLE} = \sqrt{\frac{\sum_{i=1}^{n} w_i \left(\log(1 + r_i) - \log(1 + \hat{r}_i)\right)^2}{\sum_{i=1}^{n} w_i}}
$$

where $w_i$ is the weight for rating $i$.

**Task:**

Implement `WRMSLE(y_true, y_pred, weights=None)`. If `weights` is not provided, use equal weights of 1 for all ratings.

Test your function with:

```python
weights = np.array([2.0, 1.0, 1.0, 1.5, 1.0])
```

In [ ]:
## Your solution here.

# **Q2: Bias-Based Baseline Recommenders** (15 points)

In this question, you will implement three baseline recommender systems using the Netflix dataset.

```python
import numpy as np
import pandas as pd

train = pd.read_csv('https://raw.githubusercontent.com/statmlben/CUHK-STAT3009/main/dataset/netflix/train.csv')
test = pd.read_csv('https://raw.githubusercontent.com/statmlben/CUHK-STAT3009/main/dataset/netflix/test.csv')
```

## **Q2.1: User Mean and Item Mean Baselines** (5 points)

Implement two simple baseline recommenders.

**User Mean:** For a user–item pair $(u,i)$, predict the rating as the mean of all ratings given by user $u$:

$$\widehat{r}_{ui} = \bar{r}_u = \frac{1}{|\Omega_u|} \sum_{(u,i') \in \Omega} r_{ui'}$$

where $\Omega_u$ is the set of items rated by user $u$.

**Item Mean:** For a user–item pair $(u,i)$, predict the rating as the mean of all ratings received by item $i$:

$$\widehat{r}_{ui} = \bar{r}_i = \frac{1}{|\Omega_i|} \sum_{(u',i) \in \Omega} r_{u'i}$$

where $\Omega_i$ is the set of users who rated item $i$.

**Tasks:**

1. Create `UserMeanRS` and `ItemMeanRS` classes that inherit from `sklearn.base.BaseEstimator`. Each class should have:
   - `fit(X, y)`: compute the mean rating for each user (or item)
   - `predict(X)`: return the corresponding mean for each user–item pair

2. For unseen users or items, use the global mean of the training ratings as a fallback.

3. Fit both models to the training data, generate predictions on the test data, and report the RMSE for each model. Which baseline performs better?

**Code skeleton:**

```python
class UserMeanRS(BaseEstimator):
    def fit(self, X, y):
        # X has columns: user_id, item_id
        # y is the rating
        ...
        return self

    def predict(self, X):
        ...
```

## **Q2.2: Trimmed Mean Baseline** (5 points)

A **trimmed mean** removes a fixed percentage of the smallest and largest values before computing the mean. This reduces the influence of outliers.

For a user–item pair $(u,i)$, predict:

$$\widehat{r}_{ui} = \frac{r_u^{(\text{trim})} + r_i^{(\text{trim})}}{2}$$

where $r_u^{(\text{trim})}$ is the 10% trimmed mean of ratings given by user $u$, and $r_i^{(\text{trim})}$ is the 10% trimmed mean of ratings received by item $i$.

**Tasks:**

1. Create `TrimmedMeanRS(BaseEstimator)` with parameter `trim_percent` (default `0.1`). Implement:
   - `fit(X, y)`: compute the trimmed mean for each user and each item
   - `predict(X)`: return the average of the user and item trimmed means

2. For unseen users or items, use the global trimmed mean of the training ratings. If a user or item has fewer than 10 ratings, use the regular mean instead.

3. Fit the model with `trim_percent=0.1`, generate predictions on the test data, and report the RMSE.

**Hint:** Use `scipy.stats.trim_mean()` or implement trimming with array slicing.

## **Q2.3: Harmonic Mean Baseline** (5 points)

The **harmonic mean** gives more weight to lower values. For a user–item pair $(u,i)$, predict:

$$\widehat{r}_{ui} = \frac{2}{\frac{1}{r_u^{(h)}} + \frac{1}{r_i^{(h)}}}$$

where $r_u^{(h)}$ is the harmonic mean of ratings given by user $u$, and $r_i^{(h)}$ is the harmonic mean of ratings received by item $i$:

$$r_u^{(h)} = \frac{n_u}{\sum_{(u,i') \in \Omega} \frac{1}{r_{ui'}}}, \qquad r_i^{(h)} = \frac{n_i}{\sum_{(u',i) \in \Omega} \frac{1}{r_{u'i}}}$$

**Tasks:**

1. Create `HarmonicMeanRS(BaseEstimator)`. Implement:
   - `fit(X, y)`: compute the harmonic mean for each user and each item
   - `predict(X)`: return the harmonic mean of the user and item harmonic means

2. For unseen users or items, use the global harmonic mean of the training ratings.

3. Fit the model, generate predictions on the test data, and report the RMSE.

In [ ]:
## Your solution here.

# **Q3: Matrix Factorization Models** (40 points)

## **Q3.1: SVD with Different Hyperparameters** (10 points)

Use the `SVD` class from `TabRS.py` and the Netflix dataset.

```python
import numpy as np
import pandas as pd

train = pd.read_csv('https://raw.githubusercontent.com/statmlben/CUHK-STAT3009/main/dataset/netflix/train.csv')
test = pd.read_csv('https://raw.githubusercontent.com/statmlben/CUHK-STAT3009/main/dataset/netflix/test.csv')
```

**Task:**

Train an `SVD` model with $K = 15$ and $\lambda = 0.005$ on the training data. Generate predictions on the test data and report the RMSE.

> The correctness of the implementation will be evaluated based on the code structure and logic, not on the final evaluation results.

In [ ]:
## Your solution here.

## **Q3.2: Diagnosing Underfitting and Overfitting** (5 points)

The following output is from SVD training and validation:

```text
Fitting Reg-SVD: K: 2, lam: 0.50000
Reg-SVD: ite: 0; train RMSE: 1.125
Reg-SVD: ite: 1; train RMSE: 1.098
Reg-SVD: ite: 2; train RMSE: 1.087
Reg-SVD: ite: 3; train RMSE: 1.081
Reg-SVD: ite: 4; train RMSE: 1.078
Reg-SVD: ite: 5; train RMSE: 1.076
Reg-SVD: ite: 6; train RMSE: 1.075
Reg-SVD: ite: 7; train RMSE: 1.074
Reg-SVD: ite: 8; train RMSE: 1.073
Reg-SVD: ite: 9; train RMSE: 1.073
Validation RMSE for SVD: 1.089
```

What issue is most likely occurring?

**(A)** The model is overfitting — increase regularization ($\lambda$)

**(B)** The model is underfitting — increase the number of latent factors ($K$)

**(C)** The model is underfitting — decrease regularization ($\lambda$)

**(D)** Both B and C are correct

> State your choice and explain your reasoning.

## **Q3.3: Hyperparameter Tuning with RandomizedSearchCV** (10 points)

Use the `SVD` class from `TabRS.py` and the synthetic dataset below.

```python
import pandas as pd

data = {
    'user_id': [0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10, 11, 11],
    'item_id': [0, 2, 1, 3, 0, 3, 2, 4, 1, 4, 2, 5, 3, 5, 4, 6, 5, 7, 6, 8, 7, 9, 8, 9],
    'rating': [4, 5, 5, 4, 3, 2, 4, 3, 5, 4, 3, 4, 2, 3, 4, 5, 3, 4, 4, 5, 3, 4, 5, 4]
}
df = pd.DataFrame(data)
```

**Task:**

Use `sklearn.model_selection.RandomizedSearchCV` with `cv=3` and `n_iter=10` to find the optimal hyperparameters. The parameter distributions are:

```python
from scipy.stats import randint, uniform

param_distributions = {
    'K': randint(2, 15),
    'lam': uniform(0.001, 0.1)
}
```

Report the cross-validation results and the optimal hyperparameters found.

> The correctness of the implementation will be evaluated based on the code structure and logic, not on the final evaluation results.

In [ ]:
## Your solution here.

## **Q3.4: Rank-Weighted SVD** (15 points)

In standard SVD, all latent factors are treated equally. In this task, you will implement a **rank-weighted SVD** where different latent dimensions have different importance weights.

The prediction formula is:

$$\widehat{r}_{ui} = \sum_{k=1}^{K} w_k \cdot p_{uk} \cdot q_{ik}$$

where the weight for the $k$-th latent dimension is:

$$w_k = \frac{K - k + 1}{\sum_{j=1}^{K} (K - j + 1)} = \frac{K - k + 1}{\frac{K(K+1)}{2}}$$

This gives the highest weight to the first dimension and the lowest weight to the last dimension.

**Optimization Objective:**

$$
\text{argmin}_{\mathbf{P}, \mathbf{Q}} \sum_{(u,i) \in \Omega} \left( r_{ui} - \sum_{k=1}^{K} w_k \cdot p_{uk} \cdot q_{ik} \right)^2 + \lambda \left( \|\mathbf{P}\|_F^2 + \|\mathbf{Q}\|_F^2 \right)
$$

The rank weights $w_k$ are applied **only to the loss term**; the regularization remains standard.

**ALS Implementation:**

Use Alternating Least Squares with rank-weighted updates. Let

$$
\mathbf{D} = \operatorname{diag}(\mathbf{w}),
$$

so the prediction is $\widehat{r}_{ui} = \mathbf{p}_u^\top \mathbf{D} \mathbf{q}_i$.

1. Initialize $\mathbf{P}$, $\mathbf{Q}$, and compute $\mathbf{w} = [w_1, \ldots, w_K]$.

2. **Update $\mathbf{P}$** (fix $\mathbf{Q}$): For each user $u$, let $\mathbf{Q}_u$ contain the rows of $\mathbf{Q}$ for the items in $\Omega_u$, and let $\mathbf{r}_u$ contain the corresponding ratings. Solve
   $$
   \mathbf{p}_u
   =
   \text{argmin}_{\mathbf{p}_u}
   \sum_{i \in \Omega_u}
   \left( r_{ui} - \mathbf{p}_u^\top \mathbf{D} \mathbf{q}_i \right)^2
   +
   \lambda \|\mathbf{p}_u\|^2.
   $$
   This is weighted least squares with design matrix $\mathbf{Q}_{u,w} = \mathbf{Q}_u \mathbf{D}$. Hence
   $$
   \mathbf{p}_u
   =
   \left( \mathbf{Q}_{u,w}^\top \mathbf{Q}_{u,w} + \lambda \mathbf{I} \right)^{-1}
   \mathbf{Q}_{u,w}^\top \mathbf{r}_u.
   $$

3. **Update $\mathbf{Q}$** (fix $\mathbf{P}$): For each item $i$, let $\mathbf{P}_i$ contain the rows of $\mathbf{P}$ for the users in $\Omega_i$, and let $\mathbf{r}_i$ contain the corresponding ratings. With $\mathbf{P}_{i,w} = \mathbf{P}_i \mathbf{D}$,
   $$
   \mathbf{q}_i
   =
   \left( \mathbf{P}_{i,w}^\top \mathbf{P}_{i,w} + \lambda \mathbf{I} \right)^{-1}
   \mathbf{P}_{i,w}^\top \mathbf{r}_i.
   $$

4. Iterate until convergence.

**Tasks:**

1. Create `RankWeightedSVD(BaseEstimator)` with `fit(X, y)` and `predict(X)` methods.

2. Train the model with:
   - $(K = 10, \lambda = 0.002)$
   - $(K = 15, \lambda = 0.001)$

3. Report the RMSE on the test set for both settings and compare with standard SVD from Q3.1. Does rank weighting improve performance?

**Code skeleton:**

```python
class RankWeightedSVD(BaseEstimator):
    def __init__(self, n_users, n_items, K=10, lam=0.01, iterNum=10):
        self.K = K
        self.lam = lam
        self.iterNum = iterNum
        # Initialize P, Q, weights
        ...

    def fit(self, X, y):
        # ALS with rank-weighted updates
        ...
        return self

    def predict(self, X):
        # Apply rank weights
        ...
```

> The correctness of the implementation will be evaluated based on the code structure and logic, not on the final evaluation results.

In [ ]:
## Your solution here.

# **Q4: Side Information and Neural Recommendation** (35 points)

**Dataset**

Use the *Udemy* dataset:

```python
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/statmlben/CUHK-STAT3009/refs/heads/main/dataset/udemy/udemy_clean.csv')
```

The dataset contains categorical features (`Instructor`, `Level`) and dense features (`User_vote`, `Total_hours`, `Lecture`).

## **Q4.1: Preprocessing** (10 points)

Preprocess the features:

1. Separate the features into continuous (dense) and discrete (categorical).
2. Normalize the continuous features to [0, 1] using `MinMaxScaler`.
3. Encode the discrete features using `LabelEncoder`.

Provide the preprocessed data for the next task.

In [ ]:
## Your solution here.

## **Q4.2: Factorization Machine Neural Network** (25 points)

Implement a Factorization Machine Neural Network (FM-NN) using TensorFlow/Keras.

**Mathematical Formula**

For the $i$-th record, the prediction is:

$$
\widehat{r}_i = \mathbf{w}^\top \mathbf{x}_i + \langle \mathbf{e}_{\text{Instructor},i}, \mathbf{e}_{\text{Level},i} \rangle + f_{\text{NN}}\left([\mathbf{o}_{\text{Instructor},i}; \mathbf{o}_{\text{Level},i}; \mathbf{e}_{\text{Level},i} \circ \mathbf{e}_{\text{Instructor},i}; \mathbf{x}_i]\right)
$$

**Notation:**

- $\mathbf{x}_i \in \mathbb{R}^3$: Normalized dense features (`User_vote`, `Total_hours`, `Lecture`)
- $\mathbf{e}_{\text{Instructor},i}, \mathbf{e}_{\text{Level},i} \in \mathbb{R}^8$: FM embeddings for Instructor and Level
- $\mathbf{o}_{\text{Instructor},i}, \mathbf{o}_{\text{Level},i} \in \mathbb{R}^8$: NN embeddings for Instructor and Level (separate from FM embeddings)
- $\circ$: Element-wise (Hadamard) product

**Model Components:**

1. **Linear:** $\mathbf{w}^\top \mathbf{x}_i$ (trainable weights $\mathbf{w} \in \mathbb{R}^3$)
2. **FM:** Dot product $\langle \mathbf{e}_{\text{Instructor},i}, \mathbf{e}_{\text{Level},i} \rangle$
3. **NN:** MLP applied to the concatenated vector $[\mathbf{o}_{\text{Instructor},i}; \mathbf{o}_{\text{Level},i}; \mathbf{e}_{\text{Level},i} \circ \mathbf{e}_{\text{Instructor},i}; \mathbf{x}_i]$ (dimension $8+8+8+3=27$)
   - Dense(64, activation='relu')
   - Dense(32, activation='relu')
   - Dense(1, activation='linear')

**Tasks:**

1. Implement `FM_NN` as a `Keras.Model` with the three components above.

2. Create **four embedding layers** in total:
   - `embedding_instructor_fm` (dim=8)
   - `embedding_level_fm` (dim=8)
   - `embedding_instructor_nn` (dim=8)
   - `embedding_level_nn` (dim=8)

3. Train with SGD (learning rate 0.001), maximum 50 epochs, batch size 256, and early stopping with patience 5 on validation RMSE. Use a validation split of 20%.

> The correctness of the implementation will be evaluated based on how well the code matches the formula, not on the final evaluation results.

In [ ]:
## Your solution here.